# पाठ ०३ - एजेन्टिक डिजाइन ढाँचा

यस पाठमा, हामी प्रभावकारी एआई एजेन्टहरू निर्माण गर्न तीन आधारभूत डिजाइन ढाँचाहरू अन्वेषण गर्छौं:

1. **स्पष्ट एजेन्ट निर्देशनहरू** — एजेन्ट व्यवहारलाई मार्गदर्शन गर्ने सटीक, भूमिका परिभाषित गर्ने प्रॉम्प्टहरू तयार पार्नु
2. **पाइडान्टिक मोडेलहरूसँग संरचित आउटपुट** — एजेन्टहरूले पूर्वानुमान योग्य, प्रमाणित डाटा फिर्ता गर्न सुनिश्चित गर्नु
3. **एकल जिम्मेवारी एजेन्टहरू** — केन्द्रित एजेन्टहरू डिजाइन गर्नु जसले प्रत्येकले एउटै कुरा राम्रोसँग गर्छ

हामी प्रत्येक ढाँचालाई **यात्रा गन्तव्य सिफारिसकर्ता** परिदृश्यमा लागू गर्ने छौं, क्रमिक रूपमा एउटा प्रणाली निर्माण गर्दै जसले गन्तव्यहरूको सिफारिस गर्न, उपलब्धता जाँच गर्न, र व्यवस्थापन सम्हाल्न सक्छ।


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Pattern 1: स्पष्ट एजेन्ट निर्देशनहरू

सबैभन्दा प्रभावकारी नमूना पनि सबैभन्दा सरल हो: तपाईंको एजेन्टका लागि स्पष्ट, विस्तृत निर्देशनहरू लेख्नु।

राम्रो निर्देशनले परिभाषित गर्छ:
- **को** एजेन्ट हो (व्यक्तित्व र स्वर)
- **के** यसले गर्नुपर्छ (चरण-दर-चरण जिम्मेवारीहरू)
- **कसरी** यसको व्यवहार हुनुपर्छ (सीमाहरू र शैली)

तल, हामी एक यात्रा कन्सेर्ज एजेन्ट सिर्जना गर्छौं जसले प्रत्येक प्रतिक्रिया तयार पार्न स्पष्ट निर्देशनहरू समावेश गर्दछ।


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Pydantic मोडेलहरू सहित संरचित आउटपुट

मुक्त रूपको पाठ संवादका लागि उपयोगी छ, तर डाउनस्ट्रीम प्रणालीहरूले संरचित डेटा आवश्यक पर्छ।
**Pydantic मोडेलहरू** लाई **टूल फंक्शन** सँग जोडेर, हामीले गर्न सक्छौं:

- एजेन्टको आउटपुटका लागि एकदम ठ्याक्कै स्कीमा परिभाषित गर्ने
- प्रतिक्रियाहरू स्वतः प्रमाणित गर्ने
- आवेदन तर्कमा एजेन्टका परिणामहरू विश्वसनीय रूपमा समावेश गर्ने

हामी एउटा टूल पनि परिचय गराउँछौं जुन गन्तव्य विवरण फर्काउँछ ताकि एजेन्टले आफ्नो सिफारिसहरू वास्तविक डेटा मा आधारित राख्न सकोस्।


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: एकल जिम्मेवारी एजेन्टहरू

जटिल कार्यहरूलाई धेरै फोकस गरिएको एजेन्टहरूमा विभाजन गर्दा फाइदा हुन्छ, प्रत्येकसँग एकल जिम्मेवारी हुन्छ:

- स्थान र उपलब्धताबारे जानकार एक **गन्तव्य विशेषज्ञ**
- उडान, होटेल, र यात्रा योजना बनाउने एक **लजिस्टिक्स योजनाकार**

यसले सफ्टवेयर इन्जिनियरिङ सिद्धान्त *चासोको पृथक्करण* लाई प्रतिबिम्बित गर्दछ — प्रत्येक एजेन्टलाई स्वतन्त्र रूपमा परीक्षण, मर्मत र सुधार गर्न सजिलो हुन्छ।


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## सारांश

यस पाठमा हामीले ट्राभल सिफारिस गर्ने परिदृश्यमा तीनवटा एजेन्टिक डिजाइन ढाँचा लागू गर्यौं:

| ढाँचा | प्रमुख विचार | लाभ |
|---|---|---|
| **स्पष्ट निर्देशनहरू** | पर्सोना, जिम्मेवारीहरू, र सीमाहरू पहिल्यै परिभाषित गर्नुहोस् | सुसंगत, ब्रान्डअनुकूल एजेन्ट व्यवहार |
| **संरचित आउटपुट** | पाइडान्टिक मोडेलहरू प्रतिक्रिया स्वरूप प्रयोग गर्नुहोस् | प्रमाणित, मेसिन-पठनीय नतिजाहरू |
| **एकल जिम्मेवारी** | प्रत्येक एजेन्टलाई एक केन्द्रित काम दिनुहोस् | परीक्षण गर्न, मर्मत गर्न, र संयोजन गर्न सजिलो |

यी ढाँचाहरू प्राकृतिक रूपमा संयोजन हुन्छन् — तपाईँले एकल जिम्मेवारी एजेन्ट भित्र स्पष्ट निर्देशनहरू र संरचित आउटपुट संयोजन गरी बलियो, उत्पादन-तयार प्रणालीहरू निर्माण गर्न सक्नुहुन्छ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
